# LabSD E1 — Cascade Degradation

Thin orchestrator. Heavy logic lives in the `labsd` package (Kaggle Dataset `ifty1011/labsd-src`).

Phases:
1. Env check + install
2. Build Boston/Singapore splits
3. Train C1 (CenterPoint) on Boston
4. Train C2 (MTP) on Boston
5. Set up C3 (IDM planner)
6. Baseline 5-measurement run on Singapore
7. Retrain C1 on Singapore
8. Post-retrain 5-measurement run on Singapore
9. Table I + cascade diagnostic + plot

In [ ]:
# Phase 1 — env check
import torch, os
print('CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
!ls /kaggle/input/
!df -h /kaggle/working

In [ ]:
# Phase 1 — install (skip on warm cache)
# TODO: pin torch, mmengine, mmcv, mmdet, mmdet3d, nuscenes-devkit
!pip install -q -e /kaggle/input/labsd-src/

In [ ]:
# Phase 2 — splits
from labsd.splits import build_splits
splits = build_splits()
print(splits)

In [ ]:
# Phase 3 — train C1 on Boston
from labsd.train_c1 import train_c1
c1_boston = train_c1(
    config_path='/kaggle/input/labsd-src/configs/centerpoint_boston.py',
    work_dir='/kaggle/working/c1_boston',
)
print(c1_boston)

In [ ]:
# Phase 4 — train C2 on Boston
from labsd.train_c2 import train_c2
c2_boston = train_c2(split='boston_train', out_path='/kaggle/working/c2_boston.pth')
print(c2_boston)

In [ ]:
# Phase 6 — baseline 5-measurement run on Singapore
from labsd.eval import run_all_measurements
baseline = run_all_measurements(
    c1_ckpt=c1_boston, c2_ckpt=c2_boston,
    split='singapore_val', out_path='/kaggle/working/baseline.json',
)
print(baseline)

In [ ]:
# Phase 7 — retrain C1 on Singapore
from labsd.train_c1 import train_c1
c1_singapore = train_c1(
    config_path='/kaggle/input/labsd-src/configs/centerpoint_singapore.py',
    work_dir='/kaggle/working/c1_singapore',
    resume_from=c1_boston,
)
print(c1_singapore)

In [ ]:
# Phase 8 — post-retrain 5-measurement run
after = run_all_measurements(
    c1_ckpt=c1_singapore, c2_ckpt=c2_boston,
    split='singapore_val', out_path='/kaggle/working/after_retrain.json',
)
print(after)

In [ ]:
# Phase 9 — Table I + diagnostic
from labsd.table import build_table, cascade_diagnostic
df = build_table('/kaggle/working/baseline.json',
                 '/kaggle/working/after_retrain.json',
                 '/kaggle/working/table_I.csv')
print(df.to_markdown(index=False))
print(cascade_diagnostic(df))

In [ ]:
# Phase 9 — headline plot
import matplotlib.pyplot as plt
# TODO: bar chart of C3 isolated vs C3 pipeline, before/after
plt.savefig('/kaggle/working/cascade_result.png', dpi=150, bbox_inches='tight')